# Week 11 — Query Performance & QUERY_HISTORY
**Phase 4: Capstone Prep · ~15 minutes · All tables**

---

## What you're practicing this week

Writing correct SQL is one thing — writing SQL that runs efficiently is another. This week you'll learn how to read Snowflake's query history, compare scan costs, and apply a few techniques that make a real difference to performance at scale.

## Key concepts this week

### QUERY_HISTORY
Snowflake logs every query you run. You can query this log to audit performance:

```sql
-- See your recent queries with execution time and bytes scanned
SELECT
    query_id,
    query_text,
    total_elapsed_time,  -- milliseconds
    bytes_scanned,
    execution_status
FROM TABLE(INFORMATION_SCHEMA.QUERY_HISTORY_BY_USER())
ORDER BY start_time DESC
LIMIT 20;
```

### What affects performance in Snowflake

| Technique | Why it helps |
|---|---|
| Filter early with WHERE | Snowflake skips micro-partitions that don't match — less data scanned |
| Avoid SELECT * in production | Only pull the columns you need |
| Use CTEs over repeated subqueries | CTEs are materialised once |
| Filter before joining | Reduce row counts before the join, not after |

### EXPLAIN
Shows the execution plan for a query without running it:

```sql
EXPLAIN
SELECT COUNT(*) FROM ANALYTICS_TEAM_DB.SQL_CHALLENGE.OLIST_ORDERS
WHERE order_status = 'delivered';
```

---

## Try the examples

In [ ]:
-- Example 1: view your recent query history
SELECT
    query_id,
    LEFT(query_text, 80)   AS query_preview,
    total_elapsed_time     AS elapsed_ms,
    bytes_scanned,
    execution_status
FROM TABLE(INFORMATION_SCHEMA.QUERY_HISTORY_BY_USER())
WHERE execution_status = 'SUCCESS'
ORDER BY start_time DESC
LIMIT 20;

In [ ]:
-- Example 2: run two versions of the same query, then compare scan cost in history
-- Version A: no filter
SELECT COUNT(*) AS total FROM ANALYTICS_TEAM_DB.SQL_CHALLENGE.OLIST_ORDERS;

-- Version B: filtered
SELECT COUNT(*) AS delivered FROM ANALYTICS_TEAM_DB.SQL_CHALLENGE.OLIST_ORDERS WHERE order_status = 'delivered';

In [ ]:
-- Example 3: EXPLAIN — see the execution plan without running the query
EXPLAIN
SELECT o.order_id, SUM(p.payment_value) AS total
FROM ANALYTICS_TEAM_DB.SQL_CHALLENGE.OLIST_ORDERS o
JOIN ANALYTICS_TEAM_DB.SQL_CHALLENGE.OLIST_ORDER_PAYMENTS p ON o.order_id = p.order_id
GROUP BY o.order_id;

---

## Task 1 — Find your 5 slowest queries

Using `INFORMATION_SCHEMA.QUERY_HISTORY_BY_USER()`, find your 5 slowest successful queries. Return `query_id`, `total_elapsed_time` in seconds (divide by 1000), `bytes_scanned`, and the first 100 characters of `query_text`.

> **What to think about:** Is the slowest query also the one that scanned the most bytes? They're related but not always the same.

In [ ]:
-- Task 1: your query here


In [ ]:
-- Submit Task 1
CALL ANALYTICS_TEAM_DB.SQL_CHALLENGE.SUBMIT(
    CURRENT_USER(),
    11,
    1,
    $$
        -- paste your Task 1 query here
    $$
);

---

## Task 2 — Compare filtered vs unfiltered scan cost

Run the two queries in Example 2 above if you haven't already, then query QUERY_HISTORY to compare `bytes_scanned` for each. Return both rows side by side with a label column.

> **What to think about:** In a table with billions of rows, this difference would be enormous. This is why filtering early matters.

In [ ]:
-- Task 2: your query here


In [ ]:
-- Submit Task 2
CALL ANALYTICS_TEAM_DB.SQL_CHALLENGE.SUBMIT(
    CURRENT_USER(),
    11,
    2,
    $$
        -- paste your Task 2 query here
    $$
);

---

## Task 3 — Rewrite a subquery as a CTE

Take this subquery-based query and rewrite it cleanly using CTEs:

```sql
SELECT o.order_id, o.order_status, p.total_payment
FROM ANALYTICS_TEAM_DB.SQL_CHALLENGE.OLIST_ORDERS o
JOIN (
    SELECT order_id, SUM(payment_value) AS total_payment
    FROM ANALYTICS_TEAM_DB.SQL_CHALLENGE.OLIST_ORDER_PAYMENTS
    GROUP BY order_id
) p ON o.order_id = p.order_id
WHERE p.total_payment > 200;
```

> **What to think about:** Does the CTE version feel easier to read and extend? Run both and compare execution time.

In [ ]:
-- Task 3: your query here


In [ ]:
-- Submit Task 3
CALL ANALYTICS_TEAM_DB.SQL_CHALLENGE.SUBMIT(
    CURRENT_USER(),
    11,
    3,
    $$
        -- paste your Task 3 query here
    $$
);

---

## Task 4 — Audit your own submissions

Query `ANALYTICS_TEAM_DB.SQL_CHALLENGE.SUBMISSIONS` to show all of your submissions from this challenge. Return `week_number`, `task_number`, `submitted_at`, and the first 80 characters of `query_text`. Filter to your own username using `CURRENT_USER()`.

> **What to think about:** How many re-submissions did you make on any given task? Can you see your queries improving?

In [ ]:
-- Task 4: your query here


In [ ]:
-- Submit Task 4
CALL ANALYTICS_TEAM_DB.SQL_CHALLENGE.SUBMIT(
    CURRENT_USER(),
    11,
    4,
    $$
        -- paste your Task 4 query here
    $$
);

---

## Bonus challenge

Use QUERY_HISTORY to calculate your average query execution time per week of the challenge. Which week had the most complex queries on average?

In [ ]:
-- Bonus: your query here


---

*Next week: Capstone — combining everything into one end-to-end analytics script.*